In [1]:
import os
import json
import time
import random
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from collections import defaultdict

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from torchvision import models

# Sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import (
    f1_score, accuracy_score, roc_auc_score, 
    precision_score, recall_score, classification_report
)

# ========== CONFIGURATION ==========
BASE_DIR = r"D:\Projects\CLARITY\Model\Dataset\archive"
CSV_PATH = r"D:\Projects\CLARITY\Model\Dataset\archive\Data_Entry_2017.csv"
OUTPUT_DIR = r"./outputs_best_model_resnet152"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# ========== HYPERPARAMETERS ==========
SEED = 42
BATCH_SIZE = 16
IMAGE_SIZE = 224
NUM_EPOCHS = 5
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-5

# ========== REPRODUCIBILITY ==========
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ========== DEVICE ==========
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 70)
print("CLARITY - ResNet152 Multi-Label Classification")
print("=" * 70)
print(f"BASE_DIR: {BASE_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")
print(f"Device: {DEVICE}")
print(f"PyTorch version: {torch.__version__}")
print(f"SEED: {SEED}")
print(f"Batch Size: {BATCH_SIZE}")
print(f"Image Size: {IMAGE_SIZE}")
print("=" * 70)

CLARITY - ResNet152 Multi-Label Classification
BASE_DIR: D:\Projects\CLARITY\Model\Dataset\archive
OUTPUT_DIR: ./outputs_best_model_resnet152
Device: cuda
PyTorch version: 2.7.1+cu118
SEED: 42
Batch Size: 16
Image Size: 224


In [2]:
# ================= Cell 2 =================
# Load CSV, Parse Labels, Build Metadata (CORRECTED for images_*/images/ structure)

print("\n[STEP 2] Loading and preparing dataset...")

# ========== LOAD CSV ==========
df = pd.read_csv(CSV_PATH)
print(f"✅ CSV loaded: {df.shape}")
print(df.head(3))

# ========== EXTRACT COLUMNS & PARSE LABELS ==========
df = df[['Image Index', 'Finding Labels', 'Patient ID']].copy()

# Split multi-label strings into lists
df['Finding Labels'] = df['Finding Labels'].apply(lambda x: x.split('|'))

# Get all unique disease labels
all_labels = sorted(set([l for sublist in df['Finding Labels'] for l in sublist]))

print(f"\n✅ Total unique disease labels: {len(all_labels)}")
print(f"Labels: {all_labels}")

# ========== MULTI-LABEL BINARIZATION ==========
mlb = MultiLabelBinarizer(classes=all_labels)
label_matrix = mlb.fit_transform(df['Finding Labels'])
label_cols = mlb.classes_

# Add binary columns to dataframe
df = pd.concat([df, pd.DataFrame(label_matrix, columns=label_cols)], axis=1)

print(f"\n✅ Final dataframe shape: {df.shape}")
print(f"Label columns ({len(label_cols)}): {list(label_cols)}")

# ========== BUILD IMAGE PATH CACHE (CORRECT STRUCTURE) ==========
print("\n[Building image path cache with glob...]")

img_lookup = {}

# CORRECT: Pattern for images_*/images/*.png structure
pattern = os.path.join(BASE_DIR, "images_*", "images", "*.png")
print(f"   Glob pattern: {pattern}")

all_images = glob.glob(pattern)
print(f"   Found {len(all_images)} images via glob")

for img_path in all_images:
    fname = os.path.basename(img_path)
    img_lookup[fname] = img_path

print(f"✅ Image cache built: {len(img_lookup)} images indexed")

# ========== ASSIGN IMAGE PATHS TO DATAFRAME ==========
print("\n[Mapping image paths to dataframe records...]")

img_paths = []
missing_count = 0

for idx_name in df['Image Index']:
    if idx_name in img_lookup:
        img_paths.append(img_lookup[idx_name])
    else:
        img_paths.append(None)
        missing_count += 1

df['img_path'] = img_paths

print(f"✅ Image paths assigned")
print(f"   Total records: {len(df)}")
print(f"   With valid paths: {len(df) - missing_count}")
print(f"   Missing paths: {missing_count}")

# ========== REMOVE RECORDS WITH MISSING IMAGES ==========
df_before = len(df)
df = df[df['img_path'].notna()].reset_index(drop=True)
df_after = len(df)

print(f"\n✅ After removing missing images:")
print(f"   Removed: {df_before - df_after} records")
print(f"   Remaining: {df_after} samples")

# ========== VALIDATE DATASET ==========
print(f"\n✅ Dataset prepared successfully!")
print(f"   Shape: {df.shape}")
print(f"   Patients: {df['Patient ID'].nunique()}")
print(f"   Label distribution (first 8):")

for i, label in enumerate(label_cols[:8]):
    count = df[label].sum()
    pct = 100 * count / len(df)
    print(f"      {i+1:2d}. {label:20s}: {count:6.0f} ({pct:5.1f}%)")

# ========== VERIFY SAMPLE PATHS ==========
print(f"\n[Sample path verification]")
sample_paths = df['img_path'].sample(min(5, len(df)), random_state=SEED)
for i, path in enumerate(sample_paths.values, 1):
    exists = os.path.exists(path)
    size_kb = os.path.getsize(path) / 1024 if exists else 0
    status = "✅" if exists else "❌"
    print(f"   {status} Sample {i}: {os.path.basename(path)} ({size_kb:.1f} KB)")

print("\n" + "=" * 70)


[STEP 2] Loading and preparing dataset...
✅ CSV loaded: (112120, 12)
        Image Index          Finding Labels  Follow-up #  Patient ID  \
0  00000001_000.png            Cardiomegaly            0           1   
1  00000001_001.png  Cardiomegaly|Emphysema            1           1   
2  00000001_002.png   Cardiomegaly|Effusion            2           1   

   Patient Age Patient Gender View Position  OriginalImage[Width  Height]  \
0           58              M            PA                 2682     2749   
1           58              M            PA                 2894     2729   
2           58              M            PA                 2500     2048   

   OriginalImagePixelSpacing[x     y]  Unnamed: 11  
0                        0.143  0.143          NaN  
1                        0.143  0.143          NaN  
2                        0.168  0.168          NaN  

✅ Total unique disease labels: 15
Labels: ['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Effusion', 'Emphys

In [3]:
# ================= Cell 3 =================
# Train/Val/Test Split by Patient ID (NO DATA LEAKAGE) + DataLoaders

print("\n[STEP 3] Patient-level split and DataLoader creation...")

# ========== PATIENT-LEVEL SPLITTING ==========
# Critical: Split by Patient ID to prevent data leakage
patients = df['Patient ID'].unique()
print(f"\nTotal unique patients: {len(patients)}")

# 70% train, 15% val, 15% test
train_patients, temp_patients = train_test_split(
    patients, test_size=0.3, random_state=SEED
)
val_patients, test_patients = train_test_split(
    temp_patients, test_size=0.5, random_state=SEED
)

# Create dataframes for each split
train_df = df[df['Patient ID'].isin(train_patients)].reset_index(drop=True)
val_df = df[df['Patient ID'].isin(val_patients)].reset_index(drop=True)
test_df = df[df['Patient ID'].isin(test_patients)].reset_index(drop=True)

print(f"\n✅ Patient-Level Split:")
print(f"   Train: {len(train_df):6d} images from {len(train_patients):5d} patients ({100*len(train_df)/len(df):.1f}%)")
print(f"   Val:   {len(val_df):6d} images from {len(val_patients):5d} patients ({100*len(val_df)/len(df):.1f}%)")
print(f"   Test:  {len(test_df):6d} images from {len(test_patients):5d} patients ({100*len(test_df)/len(df):.1f}%)")

# Verify no patient overlap
print(f"\n✅ Verification (no data leakage):")
print(f"   Train ∩ Val patients: {len(set(train_patients) & set(val_patients))}")
print(f"   Train ∩ Test patients: {len(set(train_patients) & set(test_patients))}")
print(f"   Val ∩ Test patients: {len(set(val_patients) & set(test_patients))}")

# ========== DATA AUGMENTATIONS ==========
train_transforms = T.Compose([
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(15),
    T.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    T.RandomResizedCrop(IMAGE_SIZE, scale=(0.85, 1.0)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225])
])

val_transforms = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225])
])

print(f"\n✅ Data augmentations configured")

# ========== CUSTOM DATASET CLASS ==========
class ChestXrayDataset(Dataset):
    """Multi-label chest X-ray dataset"""
    
    def __init__(self, dataframe, label_cols, transform=None):
        """
        Args:
            dataframe: DataFrame with image paths and labels
            label_cols: List of disease label column names
            transform: Image transformations
        """
        self.df = dataframe.reset_index(drop=True)
        self.label_cols = label_cols
        self.transform = transform
        self.img_paths = dataframe['img_path'].values
        self.labels = dataframe[label_cols].values.astype(np.float32)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # Load image
        img_path = self.img_paths[idx]
        
        try:
            image = Image.open(img_path).convert('RGB')
        except Exception as e:
            print(f"Warning: Failed to load {img_path}: {e}")
            image = Image.new('RGB', (IMAGE_SIZE, IMAGE_SIZE), color='black')
        
        # Apply transforms
        if self.transform:
            image = self.transform(image)
        
        # Get labels
        labels = torch.tensor(self.labels[idx], dtype=torch.float32)
        
        return image, labels

print(f"✅ ChestXrayDataset class defined")

# ========== CREATE DATASETS ==========
train_dataset = ChestXrayDataset(train_df, label_cols, transform=train_transforms)
val_dataset = ChestXrayDataset(val_df, label_cols, transform=val_transforms)
test_dataset = ChestXrayDataset(test_df, label_cols, transform=val_transforms)

print(f"\n✅ Datasets created:")
print(f"   Train: {len(train_dataset)} samples")
print(f"   Val:   {len(val_dataset)} samples")
print(f"   Test:  {len(test_dataset)} samples")

# ========== CREATE DATALOADERS ==========
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
    drop_last=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print(f"\n✅ DataLoaders created:")
print(f"   Train batches: {len(train_loader)} (batch size: {BATCH_SIZE})")
print(f"   Val batches:   {len(val_loader)}")
print(f"   Test batches:  {len(test_loader)}")

# ========== VERIFY SAMPLE BATCH ==========
print(f"\n[Verifying sample batch]")
sample_images, sample_labels = next(iter(train_loader))

print(f"✅ Sample batch shapes:")
print(f"   Images: {sample_images.shape} (expected: [{BATCH_SIZE}, 3, {IMAGE_SIZE}, {IMAGE_SIZE}])")
print(f"   Labels: {sample_labels.shape} (expected: [{BATCH_SIZE}, {len(label_cols)}])")
print(f"   Image range: [{sample_images.min():.3f}, {sample_images.max():.3f}]")
print(f"   Label range: [{sample_labels.min():.1f}, {sample_labels.max():.1f}]")
print(f"   Positive labels in batch: {(sample_labels > 0.5).sum().item()} / {sample_labels.numel()}")

print("\n" + "=" * 70)


[STEP 3] Patient-level split and DataLoader creation...

Total unique patients: 30805

✅ Patient-Level Split:
   Train:  78566 images from 21563 patients (70.1%)
   Val:    17063 images from  4621 patients (15.2%)
   Test:   16491 images from  4621 patients (14.7%)

✅ Verification (no data leakage):
   Train ∩ Val patients: 0
   Train ∩ Test patients: 0
   Val ∩ Test patients: 0

✅ Data augmentations configured
✅ ChestXrayDataset class defined

✅ Datasets created:
   Train: 78566 samples
   Val:   17063 samples
   Test:  16491 samples

✅ DataLoaders created:
   Train batches: 4910 (batch size: 16)
   Val batches:   1067
   Test batches:  1031

[Verifying sample batch]
✅ Sample batch shapes:
   Images: torch.Size([16, 3, 224, 224]) (expected: [16, 3, 224, 224])
   Labels: torch.Size([16, 15]) (expected: [16, 15])
   Image range: [-2.118, 2.640]
   Label range: [0.0, 1.0]
   Positive labels in batch: 21 / 240



In [4]:
# ================= Cell 4 =================
# Model Definition: ResNet152 for Multi-Label Classification
# *** ONLY CHANGE: DenseNet121 → ResNet152 ***

print("\n[STEP 4] Creating ResNet152 model...")

# ========== LOAD PRETRAINED RESNET152 ==========
print("   Loading ResNet152 (this may take ~30-60 seconds)...")
model = models.resnet152(weights=models.ResNet152_Weights.IMAGENET1K_V1)
print("   ✅ Loaded")

# Modify classifier for multi-label classification (15 diseases)
# ResNet152 has fc layer (not classifier like DenseNet)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, len(label_cols))

# Move to device
model = model.to(DEVICE)

print(f"✅ ResNet152 model created and moved to {DEVICE}")
print(f"   Total parameters: {sum(p.numel() for p in model.parameters()):,}")

# ========== VERIFY OUTPUT SHAPE (NO DATALOADER CALL) ==========
print(f"\n✅ Testing forward pass...")
with torch.no_grad():
    test_input = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE).to(DEVICE)
    test_output = model(test_input)
    print(f"   Input shape: {test_input.shape}")
    print(f"   Output shape: {test_output.shape}")
    print(f"   Expected output shape: [1, {len(label_cols)}]")
    assert test_output.shape == (1, len(label_cols)), "Output shape mismatch!"

# ========== LOSS FUNCTION ==========
criterion = nn.BCEWithLogitsLoss()
print(f"\n✅ Loss function: BCEWithLogitsLoss")

# ========== OPTIMIZER ==========
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-5

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

print(f"✅ Optimizer: AdamW (LR={LEARNING_RATE}, WD={WEIGHT_DECAY})")

# ========== SCHEDULER ==========
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=NUM_EPOCHS,
    eta_min=1e-6
)

print(f"✅ Scheduler: CosineAnnealingLR (T_max={NUM_EPOCHS})")

# ========== SUMMARY ==========
print("\n" + "=" * 70)
print("TRAINING CONFIGURATION READY")
print("=" * 70)
print(f"Model:             ResNet152 (ImageNet pretrained)")
print(f"Classes:           {len(label_cols)} (multi-label)")
print(f"Batch size:        {BATCH_SIZE}")
print(f"Epochs:            {NUM_EPOCHS}")
print(f"Train batches:     {len(train_loader)}")
print(f"Val batches:       {len(val_loader)}")
print(f"Device:            {DEVICE}")
print(f"Loss:              BCEWithLogitsLoss")
print(f"Optimizer:         AdamW")
print(f"Scheduler:         CosineAnnealingLR")
print("=" * 70 + "\n")

print("✅ CELL 4 COMPLETE - Ready for training!")


[STEP 4] Creating ResNet152 model...
   Loading ResNet152 (this may take ~30-60 seconds)...
   ✅ Loaded
✅ ResNet152 model created and moved to cuda
   Total parameters: 58,174,543

✅ Testing forward pass...
   Input shape: torch.Size([1, 3, 224, 224])
   Output shape: torch.Size([1, 15])
   Expected output shape: [1, 15]

✅ Loss function: BCEWithLogitsLoss
✅ Optimizer: AdamW (LR=0.0001, WD=1e-05)
✅ Scheduler: CosineAnnealingLR (T_max=5)

TRAINING CONFIGURATION READY
Model:             ResNet152 (ImageNet pretrained)
Classes:           15 (multi-label)
Batch size:        16
Epochs:            5
Train batches:     4910
Val batches:       1067
Device:            cuda
Loss:              BCEWithLogitsLoss
Optimizer:         AdamW
Scheduler:         CosineAnnealingLR

✅ CELL 4 COMPLETE - Ready for training!


In [5]:
# ================= Cell 5 =================
# Metrics, Training Utilities, and Main Training Loop

print("\n[STEP 5] Defining metrics and starting training...")

# ------------------------------------------------
# Utility Functions
# ------------------------------------------------
def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

def calculate_metrics(y_true, y_pred_logits, threshold=0.5):
    """Compute multi-label metrics"""
    y_pred = sigmoid(y_pred_logits)
    y_pred_bin = (y_pred >= threshold).astype(int)
    try:
        auc = roc_auc_score(y_true, y_pred, average='micro')
    except:
        auc = 0.0
    f1 = f1_score(y_true, y_pred_bin, average='micro', zero_division=0)
    precision = precision_score(y_true, y_pred_bin, average='micro', zero_division=0)
    recall = recall_score(y_true, y_pred_bin, average='micro', zero_division=0)
    return {"auc": auc, "f1": f1, "precision": precision, "recall": recall}

# ------------------------------------------------
# Training Loop
# ------------------------------------------------
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    all_preds, all_targets = [], []
    pbar = tqdm(loader, desc="Training", leave=False)
    
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        all_preds.append(outputs.detach().cpu().numpy())
        all_targets.append(labels.cpu().numpy())

        pbar.set_postfix(loss=f"{loss.item():.4f}")
    
    y_true = np.concatenate(all_targets)
    y_pred = np.concatenate(all_preds)
    metrics = calculate_metrics(y_true, y_pred)
    return total_loss / len(loader), metrics


def validate_one_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_preds, all_targets = [], []

    with torch.no_grad():
        pbar = tqdm(loader, desc="Validating", leave=False)
        for images, labels in pbar:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            all_preds.append(outputs.cpu().numpy())
            all_targets.append(labels.cpu().numpy())

            pbar.set_postfix(loss=f"{loss.item():.4f}")
    
    y_true = np.concatenate(all_targets)
    y_pred = np.concatenate(all_preds)
    metrics = calculate_metrics(y_true, y_pred)
    return total_loss / len(loader), metrics


# ------------------------------------------------
# Training Configuration
# ------------------------------------------------
best_auc = 0.0
patience = 3
no_improve = 0
history = {"train_loss": [], "val_loss": [], "train_auc": [], "val_auc": []}

print("\n" + "=" * 70)
print("STARTING TRAINING")
print("=" * 70)

for epoch in range(NUM_EPOCHS):
    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}")
    start_time = time.time()

    train_loss, train_metrics = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
    val_loss, val_metrics = validate_one_epoch(model, val_loader, criterion, DEVICE)
    scheduler.step()

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_auc"].append(train_metrics["auc"])
    history["val_auc"].append(val_metrics["auc"])

    elapsed = time.time() - start_time
    print(f"Epoch {epoch+1} completed in {elapsed/60:.1f} min")
    print(f"  Train Loss: {train_loss:.4f} | Train AUC: {train_metrics['auc']:.4f} | F1: {train_metrics['f1']:.4f}")
    print(f"  Val Loss:   {val_loss:.4f} | Val AUC:   {val_metrics['auc']:.4f} | F1: {val_metrics['f1']:.4f}")

    # Best Model Checkpoint
    if val_metrics["auc"] > best_auc:
        best_auc = val_metrics["auc"]
        no_improve = 0
        torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, f"best_resnet152_auc_{best_auc:.4f}.pth"))
        print(f"✅ Saved new best model — Val AUC: {best_auc:.4f}")
    else:
        no_improve += 1
        print(f"⚠️ No improvement ({no_improve}/{patience})")

    if no_improve >= patience:
        print("\n⛔ Early stopping triggered.")
        break

print("\n" + "=" * 70)
print(f"TRAINING COMPLETE — Best Validation AUC: {best_auc:.4f}")
print(f"Model saved in {OUTPUT_DIR}")
print("=" * 70)

# Optional: Save loss/metrics history for visualization
with open(os.path.join(OUTPUT_DIR, "training_history.json"), "w") as f:
    json.dump(history, f, indent=4)
print("✅ Training history saved")


[STEP 5] Defining metrics and starting training...

STARTING TRAINING

Epoch 1/5


Epoch 1 completed in 87.0 min
  Train Loss: 0.1927 | Train AUC: 0.8848 | F1: 0.4520
  Val Loss:   0.1852 | Val AUC:   0.8967 | F1: 0.4631
✅ Saved new best model — Val AUC: 0.8967

Epoch 2/5


Epoch 2 completed in 87.0 min
  Train Loss: 0.1825 | Train AUC: 0.9011 | F1: 0.4756
  Val Loss:   0.1793 | Val AUC:   0.9070 | F1: 0.4973
✅ Saved new best model — Val AUC: 0.9070

Epoch 3/5


Epoch 3 completed in 87.4 min
  Train Loss: 0.1771 | Train AUC: 0.9088 | F1: 0.4920
  Val Loss:   0.1763 | Val AUC:   0.9118 | F1: 0.5035
✅ Saved new best model — Val AUC: 0.9118

Epoch 4/5


Epoch 4 completed in 86.8 min
  Train Loss: 0.1720 | Train AUC: 0.9158 | F1: 0.5064
  Val Loss:   0.1746 | Val AUC:   0.9143 | F1: 0.5189
✅ Saved new best model — Val AUC: 0.9143

Epoch 5/5


Epoch 5 completed in 85.7 min
  Train Loss: 0.1679 | Train AUC: 0.9210 | F1: 0.5201
  Val Loss:   0.1720 | Val AUC:   0.9172 | F1: 0.5178
✅ Saved new best model — Val AUC: 0.9172

TRAINING COMPLETE — Best Validation AUC: 0.9172
Model saved in ./outputs_best_model_resnet152
✅ Training history saved
